In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [5]:
html_text = requests.get('https://huggingface.co/languages').text
soup = BeautifulSoup(html_text, 'html.parser')


rows = soup.find_all('tr')

# Initialize lists to store data
iso_codes = []

for row in rows:
    cells = row.find_all('td')
    if len(cells) > 0:  # Exclude header row
        iso_codes.append(cells[1].find('code').get_text(strip=True))

In [7]:
html_content = requests.get('https://huggingface.co/models?sort=trending')
soup = BeautifulSoup(html_content.text, 'html.parser')

# Find all divs with class="mb-3"
mb3_divs = soup.find_all(class_="mb-3")

# Initialize lists to store results
href_list = []

for div in mb3_divs:    
    # Find all "a" tags within the current mb-3 div
    hrefs = div.find_all('a')
    # Extract href attributes from each "a" tag and append to the list
    href_list.extend([a['href'] for a in hrefs])

In [10]:
models_names = [s.replace('/models?pipeline_tag=', '') for s in href_list]

In [12]:
def get_models_json(pipeline_tag, language):
    # URL components
    scheme = "https"
    host = "huggingface.co"
    filename = "/models-json"

    # Query parameters
    query_params = {
        "pipeline_tag": pipeline_tag,
        "language": language,
        "sort": "trending"
    }

    # Construct the full URL
    url = f"{scheme}://{host}{filename}"

    # Send GET request with query parameters
    response = requests.get(url, params=query_params)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse JSON response and return it
        return response.json()
    else:
        # Print error message if request failed
        print("Failed to retrieve data. Status code:", response.status_code)
        return None

In [13]:
trial_iso_ = iso_codes[0:20]

In [14]:
trial_models_ = models_names[0:10]

In [60]:
# Define lists of languages and pipeline tags
languages = ["en", "fr", "de"]  # Example list of languages
pipeline_tags = ["text-to-image", "image-to-text", "document-question-answering"]  # Example list of pipeline tags

def get_data_for_combinations(languages, pipeline_tags):
    # Initialize an empty list to store the results
    results = []

    # Iterate over each language and pipeline tag combination
    for language in languages:
        for pipeline_tag in pipeline_tags:
            # Call the get_models_json function for the current combination
            response_data = get_models_json(pipeline_tag, language)
            if response_data:
                # Append the results to the list
                results.append({"language": language, "pipeline_tag": pipeline_tag, "response": response_data})

    # Create a DataFrame from the results list
    df = pd.DataFrame(results)

    return df

# Example usage:
result_df = get_data_for_combinations(languages, pipeline_tags)
print(result_df)

  language                 pipeline_tag  \
0       en                text-to-image   
1       en                image-to-text   
2       en  document-question-answering   
3       fr                text-to-image   
4       fr                image-to-text   
5       fr  document-question-answering   
6       de                text-to-image   
7       de                image-to-text   
8       de  document-question-answering   

                                            response  
0  [activeFilters, models, numItemsPerPage, numTo...  
1  [activeFilters, models, numItemsPerPage, numTo...  
2  [activeFilters, models, numItemsPerPage, numTo...  
3  [activeFilters, models, numItemsPerPage, numTo...  
4  [activeFilters, models, numItemsPerPage, numTo...  
5  [activeFilters, models, numItemsPerPage, numTo...  
6  [activeFilters, models, numItemsPerPage, numTo...  
7  [activeFilters, models, numItemsPerPage, numTo...  
8  [activeFilters, models, numItemsPerPage, numTo...  


In [48]:
result_df['models_count'] = result_df['response'].apply(lambda x: len(x['models']))

In [49]:
result_df

,language,pipeline_tag,response,models_count
0,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
1,en,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,30
2,en,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,23
3,fr,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,21
4,fr,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,18
5,fr,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0
6,de,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,3
7,de,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,9
8,de,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0


# nice, but english has way more than 30 models! Iterate over pages so we can get them all 

In [62]:
languages = ["en", "fr", "de"]  # Example list of languages
pipeline_tags = ["text-to-image", "image-to-text", "document-question-answering"] 

In [73]:

def get_models_json(pipeline_tag, language, page=None):
    # URL components
    scheme = "https"
    host = "huggingface.co"
    filename = "/models-json"

    # Query parameters
    query_params = {
        "pipeline_tag": pipeline_tag,
        "language": language,
        "sort": "trending",
    }
    if page is not None:
        query_params["p"] = page

    # Construct the full URL
    url = f"{scheme}://{host}{filename}"

    # Send GET request with query parameters
    response = requests.get(url, params=query_params)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse JSON response and return it
        return response.json()
    else:
        # Print error message if request failed
        print("Failed to retrieve data. Status code:", response.status_code)
        return None 


# Define lists of languages and pipeline tags
languages = ["en", "fr", "de"]  # Example list of languages
pipeline_tags = ["text-to-image", "image-to-text", "document-question-answering"]  # Example list of pipeline tags

def get_data_for_combinations(languages, pipeline_tags):
    # Initialize an empty list to store the results
    results = []

    # Iterate over each language and pipeline tag combination
    for language in languages:
        for pipeline_tag in pipeline_tags:
            page = None
            while True:
                # Call the get_models_json function for the current combination and page
                response_data = get_models_json(pipeline_tag, language, page=page)
                if response_data:
                    # Append the results to the list
                    results.append({"language": language, "pipeline_tag": pipeline_tag, "response": response_data})
                    # Check if there are more pages
                    if len(response_data['models']) == 30:
                        page = 1 if page is None else page + 1
                        print(page)
                    else:
                        break
                else:
                    break  # Exit loop if there's an error

    # Create a DataFrame from the results list
    df = pd.DataFrame(results)

    return df

In [91]:
result_df = get_data_for_combinations(languages, pipeline_tags)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
1
2
3
4
5
6
7


In [92]:
result_df

,language,pipeline_tag,response
0,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
1,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
2,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
3,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
4,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
...,...,...,...
76,fr,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...
77,fr,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...
78,de,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...
79,de,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...


In [93]:
result_df['models_count'] = result_df['response'].apply(lambda x: len(x['models']))

In [94]:
result_df

,language,pipeline_tag,response,models_count
0,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
1,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
2,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
3,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
4,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30
...,...,...,...,...
76,fr,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,18
77,fr,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0
78,de,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,3
79,de,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,9


In [95]:
result_df['sum'] = result_df.groupby(['language', 'pipeline_tag'])['models_count'].transform('sum')

In [96]:
result_df

,language,pipeline_tag,response,models_count,sum
0,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
1,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
2,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
3,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
4,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
...,...,...,...,...,...
76,fr,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,18,18
77,fr,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0,0
78,de,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,3,3
79,de,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,9,9


In [97]:
df = result_df.drop_duplicates(subset=['language', 'pipeline_tag'])

In [98]:
df

,language,pipeline_tag,response,models_count,sum
0,en,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,30,1978
66,en,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,30,222
74,en,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,23,23
75,fr,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,21,21
76,fr,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,18,18
77,fr,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0,0
78,de,text-to-image,{'activeFilters': {'pipeline_tag': ['text-to-i...,3,3
79,de,image-to-text,{'activeFilters': {'pipeline_tag': ['image-to-...,9,9
80,de,document-question-answering,{'activeFilters': {'pipeline_tag': ['document-...,0,0


# tadannnn